# Reproducing the Toth et al. 2025 Florida Keys Coral Restoration SfM Pipeline

**Author:** Frank Velez
**Date:** May 2026
**Kernel:** Python (reef-sfm-mote-keys)
**Data:** USGS data releases P1WHKTRD (Johnson et al. 2025, raw imagery) and P13HMEON (Toth et al. 2025a, derived products)
**Source:** [USGS Coastal and Marine Geology Program](https://cmgds.marine.usgs.gov/) · [Toth et al. 2025, *Scientific Reports*](https://doi.org/10.1038/s41598-025-04818-3)
**Citation:** Johnson SM, Toth LT, Kelly EA, Smiley NA, Sullivan-Stack J, et al. (2025). *Lower Florida Keys coral reef restoration sites imagery, 2019–2023* [Data release]. U.S. Geological Survey. https://doi.org/10.5066/P1WHKTRD

***

## Project Motivation & Dataset Rationale

When I started looking for a SfM project to anchor my transition into marine science, I had a simple question: where is the actual operational work happening on coral restoration in the United States, and what data and methods are the people doing that work using? I'm coming from 15 years in clinical research informatics — comfortable with Python, biomedical data engineering, cloud workflows, provenance, and the kind of data discipline that NIH-funded studies demand — but I am genuinely new to marine science. I wanted to learn the field by doing real work in it, not by reading about it from a distance.

The question led me to Mote Marine Laboratory in the Florida Keys (I was biased as I've been to there coral nurseries), then to their published collaboration with the USGS St. Petersburg Coastal and Marine Science Center, and then to the paper that changed how I thought about this project: Toth et al. (2025), *Coral restoration can drive rapid increases in reef-accretion potential*, published in *Scientific Reports*. Alongside it I found Combs et al. (2021), the open-access *PLOS ONE* methods paper that established the underlying Structure-from-Motion (SfM) photogrammetry workflow Mote and USGS use. And alongside *those*, I found the actual data: two USGS open-access releases (P1WHKTRD and P13HMEON) containing 39,840 raw images from eleven Lower Florida Keys reef sites plus the full set of derived SfM products and topographic complexity measurements.

That combination — open methods, open imagery, open derived products, and a research team actively using all of it — is unusual, and it told me something important. The Mote/USGS group has done the hard work of making their reef restoration monitoring pipeline reproducible in principle. The pieces are all available; an outsider can in theory pick up exactly what they're doing and run it.

What I noticed reading the papers carefully, though, was the gap between *reproducible in principle* and *reproducible in practice*. The pipeline produces orthomosaics, digital elevation models, dense point clouds, and metrics like rugosity and fractal dimension from underwater image sets. But the operational data management around the pipeline — the parts a clinical informatician would expect by default — isn't formalized in the published workflow. There's no automated intake validation. The processing parameters used for any given reconstruction live in a brittle HTML report. Quality targets exist in prose in the methods papers but aren't checked programmatically. Comparing a new reconstruction against published reference values is something you'd have to script by hand, every time.

I am, again, new to marine science. I am not new to the question of what data infrastructure a serious research program needs. And the gap I was seeing — between the actual scientific work, which is excellent, and the data management discipline around it — is the kind of thing I can genuinely contribute to.

That insight reframed the project for me. This isn't just a "learn SfM photogrammetry" exercise. The pipeline is well-documented; I can learn it. The interesting question is whether someone with my background, applying the data engineering instincts I already have, can produce something genuinely useful to the team whose methods I'm learning from.

So that's the project. Reproduce the Toth et al. 2025 SfM pipeline on a single reef site (EasternDryRocks, chosen not only because it's the smallest offshore site and the cleanest data case, but also because I've been to the area alot, Christ of the Abyss, etc). Hit the same quality targets the published methodology specifies. Compare the resulting metrics against the published reference values in P13HMEON. And, while doing that, build a Python provenance and QC layer around the whole workflow — intake validation, processing manifest capture, quality-target checking, metric reconciliation — that the original pipeline doesn't have. Publish the result as a Quarto page and a public GitHub repository. Then, with v1 in hand, maybe reach out to the people whose work this builds on.

## What I'm Building

A reproducible end-to-end implementation of the Mote/USGS SfM workflow on EasternDryRocks, plus a Python layer that adds the data management discipline I'd expect around any production scientific pipeline. Concretely:

- **Cloud infrastructure** — a stable AWS EC2 GPU instance (g6.4xlarge with NVIDIA L4) running Ubuntu, configured according to an 8-step stability pattern that preserves the Metashape license fingerprint across stop/start cycles. The infrastructure itself is version-controlled bash, not click-ops.
- **Data acquisition** — Python scripts that pull the EasternDryRocks image subset from the USGS Imagery Data System, verify integrity, compute provenance hashes, and produce an intake QC report against documented expectations from the USGS metadata.
- **Metashape processing** — the SfM pipeline itself, run through Agisoft Metashape Professional following the methods in Combs 2021 and Toth 2025. Parameter values where ambiguous are taken from the NOAA PIFSC SOP (Torres-Pulliza et al. 2024), which I'm using as a parameter reference rather than as the methodological basis. The output is a complete set of products: sparse cloud, dense cloud, mesh, DEM at 0.001 m resolution, orthomosaic at 0.0005 m resolution, camera poses, scale bar error metrics, and a processing report.
- **The Python provenance and QC package** — the part I think is actually new. Four modules: intake validator, processing manifest parser (turning Metashape's HTML report into structured queryable JSON/YAML), QC validator against the SOP's documented quality targets (reprojection error 0.3–0.5 px, scale bar error ≤0.001 m, registered images ≥90%, reconstruction uncertainty ≤15), and a metric reconciliation module that computes the topographic complexity metrics from Toth 2025 on my reconstruction and compares them to the published values for the same site. Production-quality Python — type hints, tests, structured logging, CLI entry points, the works.
- **GIS annotation** — one transect annotated in QGIS for benthic cover, with publication-quality figure exports for the writeup.
- **Quarto writeup** — Written for the people whose work I'm building on, and to be genreally more accessible than a notebook.

## What This Is, and What This Isn't

This is not a research study. It's one site, processed once, by someone learning the field. It won't produce a novel ecological finding. It won't change anyone's understanding of Florida Keys coral restoration.

This is a portfolio piece that takes a published, openly available scientific workflow and adds the data management instrumentation the workflow currently lacks. The contribution, if there is one, is the instrumentation — and the demonstration that a senior data engineer transitioning into marine science can do work that's genuinely useful to the field, not just adjacent to it.

I want to be clear that I'm not coming into this assuming my outside perspective is automatically valuable. Mote and USGS have been doing this work for years; they understand things I don't, and I'm sure there are reasons certain parts of the pipeline are the way they are that I'll only learn by talking to people on those teams. What I am claiming is that the specific instrumentation gap — provenance, QC, reconciliation — is something my background prepares me to address. Whether the resulting code is useful to anyone is a question I'll answer by asking, after v1 ships.

## Methodological Lineage

The work follows two papers as its methodological basis: Combs et al. (2021), the open-access *PLOS ONE* paper establishing the SfM workflow for Florida Keys restoration monitoring, and Toth et al. (2025), the *Scientific Reports* paper applying that workflow to evaluate restoration-driven changes in reef-accretion potential. The data comes from two paired USGS data releases: P1WHKTRD (raw imagery, 2019–2023) and P13HMEON (derived SfM products and topographic complexity measurements).

The NOAA PIFSC SOP (Torres-Pulliza et al. 2024, DOI 10.25923/cydj-z260) is referenced for specific Metashape parameter values where the methods papers are less prescriptive. It is the operational standard for the *Pacific* NCRMP program, not the Atlantic / Florida Keys workflow, and I'm using it the way you'd use any well-documented parameter reference — as a source of defensible defaults — not as the methodology I'm reproducing. I want that distinction clear because conflating the two would confuse anyone in this community reading the work.

## Architecture

**Local:** MacBook (A18 Pro, macOS, 8 GB RAM). Used for editing, Git, Quarto authoring, and SSH/NICE DCV access to the cloud workstation.

**Cloud:** Single stable AWS EC2 g6.4xlarge instance with NVIDIA L4 GPU, Ubuntu Server 24.04 LTS via AWS Deep Learning AMI. Accessed primarily by SSH from the MacBook terminal; NICE DCV is used only when the Metashape GUI is required (error reduction, marker placement, manual review) and for the QGIS annotation work.

**Software:** Agisoft Metashape Professional (30-day trial first, then node-locked perpetual license after the trial validates the workflow), Python via uv-managed virtual environment, Jupyter for analysis notebooks, QGIS LTR for GIS annotation, Quarto for the writeup, Cursor with Remote-SSH for development. The instance is stopped between work sessions to minimize cost while preserving the license fingerprint.

## Build Plan

The project is organized as nine focused work sessions, each scoped to one layer:

1. Repository scaffold
2. AWS infrastructure setup
3. EC2 software bootstrap and Metashape trial activation
4. USGS data acquisition and intake validation
5. Metashape SOP-aligned processing
6. Python provenance, QC, and metric reconciliation package
7. QGIS benthic annotation and publication figures
8. Quarto writeup (this page)
9. Site publication, code cleanup, and outreach drafts to Toth, Combs, and Mote

Estimated total effort: 6–8 weekends to v1. Metashape Professional trial activates in session 3 and must produce real outputs by day 30, which constrains the middle of the build but not the writeup.

## What I Expect to Learn

Honestly, a lot. Here's a partial list, kept honest:

- How real underwater image acquisition for SfM differs from terrestrial photogrammetry in ways that matter for the processing pipeline (caustics, color attenuation with depth, surge-induced motion, the role of scale bars in absence of GPS)
- Which parts of the Mote/USGS workflow are well-served by automation and which legitimately require an experienced human eye (I suspect error reduction and scale bar marker placement are the latter)
- Whether my assumptions about data management gaps in the pipeline survive contact with the actual pipeline — possibly the people doing this work have solved these problems in ways the published methods don't describe
- What the real bottlenecks are in operational reef monitoring at scale, which I won't see from outside but might glimpse by working with the data carefully
- How to talk and write about reef science at a level that's accurate without overclaiming expertise I don't have

I'll add to this list as I go, and the published writeup will include an honest "what I learned" section at the end — including the parts where I was wrong about something at the start.

## References

Combs, I. R., Studivan, M. S., Eckert, R. J., & Voss, J. D. (2021). Quantifying impacts of stony coral tissue loss disease on corals in Southeast Florida through surveys and 3D photogrammetry. *PLOS ONE*, 16(8), e0252593. https://doi.org/10.1371/journal.pone.0252593

Johnson, S. M., Toth, L. T., et al. (2025). *Lower Florida Keys coral reef restoration sites imagery, 2019–2023* [Data release]. U.S. Geological Survey. https://doi.org/10.5066/P1WHKTRD

Toth, L. T., Johnson, S. M., et al. (2025). Coral restoration can drive rapid increases in reef-accretion potential. *Scientific Reports*, 15. https://doi.org/10.1038/s41598-025-04818-3

Toth, L. T., Johnson, S. M., et al. (2025a). *Coral reef restoration SfM products and topographic complexity data, Lower Florida Keys, 2019–2023* [Data release]. U.S. Geological Survey. https://doi.org/10.5066/P13HMEON

Torres-Pulliza, D., Charendoff, J., Couch, C. S., et al. (2024). *Processing Coral Reef Imagery Using Structure-from-Motion Photogrammetry: Standard Operating Procedures (2023 Update).* NOAA Technical Memorandum NOAA-TM-NMFS-PIFSC-159. https://doi.org/10.25923/cydj-z260

***

*Project scoped May 2026 through extended planning sessions with Anthropics Claude. Last updated: 2026-05-17.*

## Session One — Repository Scaffold

**Date:** May 18 2026
**Model used:** Claude Sonnet 4.6
**Status:** Complete
**Time:** a couple hours

The first session was deliberately mechanical: get the repository skeleton in place, push to GitHub, and end up with an empty-but-correct scaffold that the next eight sessions land into. No AWS, no Metashape, no data — just the structural decisions that everything downstream depends on.

I went into this one with low ambition on purpose. After a few long architecting sessions where the project shape kept shifting (terrestrial COLMAP → Pacific NOAA → Florida Keys USGS/Mote, cloud strategy revised several times, Linux vs Windows, license tier decisions), I wanted the first execution session to just commit something to disk. Momentum matters more than completeness on session one.

**What got built:**

- Local repository at `~/code/reef-sfm-mote-keys/`, initialized with Git and pushed to `github.com/velezf/reef-sfm-mote-keys`
- Directory structure matching the three-stage workflow document: `data/raw`, `data/processed`, `notebooks`, `scripts`, `figures`, `docs`, plus `src/reef_sfm_provenance/` for the Python package that gets built in Session 6
- `pyproject.toml` managed by uv, with a minimal initial dependency set (jupyter, ipykernel, pandas, numpy, matplotlib, pillow, pyyaml) — the philosophy is to add packages as I actually need them, not to front-load
- `.python-version` pinned, `.gitignore` matching the conventions from my workflow document (the rule about gitignoring `data/raw/*` except `.gitkeep` is the one I'll forget if I don't have it written down)
- `references.bib` prepopulated with the methodological references (Combs 2021, Toth 2025, both USGS data releases, Schönberger's SfM technical refs, Bayley & Mogg 2020, Burns et al. 2015) and the PIFSC SOP tagged in a comment as "parameter reference only, not methodological basis" — that distinction matters and I want it visible in the bib itself
- `README.md` written for the right audience: senior-technical, framed as a reproducible reimplementation and a learning exercise.
- `project-plan.md` at the repo root containing the full nine-session plan, so future sessions can reference it directly.
- Initial commit, pushed to GitHub

**What I noticed:**

The README was the part that took me longest, and it's worth saying why. The default voice my background pushes me toward is the clinical research informatics register — heavy on systems language, data governance, FAIR principles, regulatory framing. That voice is correct for the audience of a NIH grant report and wrong for the audience of a reef conservation lab. Mote and USGS readers likely want to see that I understand *their* domain language too: "topographic complexity," "belt transect," "Mission: Iconic Reefs," "outplant survival." I rewrote the README three times to find a register that's senior-technical without being clinical-technical. I think this is going to be an ongoing calibration through the project.

Citation discipline came up earlier than expected. The PIFSC SOP came into the project knowledge during my architecting conversation, and I initially talked about it as if it were the methodology I'm reproducing. It isn't — Combs 2021 + Toth 2025 are the Atlantic/Florida Keys methods; the PIFSC SOP is the Pacific operational standard. I'm using it as a parameter reference because the methods papers are less prescriptive about specific Metashape dialog values, but mixing those two roles up in the writeup would confuse a careful reviewer immediately. Catching this at Session 1, before it ends up baked into Quarto prose, was a small but important course correction. I added a comment in `references.bib` so future-me doesn't lose the distinction.

The directory structure exercise made me think about what's reusable vs what's project-specific. The Python package under `src/` is structured so the intake validator, manifest parser, QC validator, and reconciliation modules could in principle be lifted out and used by another restoration program running similar SfM workflows. I don't know yet whether anyone will actually do that. But scaffolding the code that way costs nothing extra at Session 1 and makes the eventual Session 6 deliverable look more like a tool than a one-off script. That's a small framing choice with outsized downstream effect.

**What I didn't do, on purpose:**

I didn't add any code beyond what the scaffold required. No notebook stubs, no script skeletons, no placeholder modules. The temptation was there — once you're in the repo it's easy to keep going — but the discipline of "Session N only delivers what Session N's prompt says" is something I want to hold to. Scope creep across sessions is the single biggest risk to a 6–8 weekend timeline.

I also didn't activate the Metashape trial. That's deliberate and important. The trial is 30 days from activation, and it doesn't activate until Session 3 after the cloud environment is fully bootstrapped. Burning trial days on scaffolding work would be a real and avoidable mistake.

**What's next:**

Session 2 is AWS infrastructure setup. That's the session with the most real operational risk — license fingerprint stability across stop/start cycles, secondary ENI configuration, EBS snapshot semantics, cost monitoring. I'm switching from Sonnet to Opus for Session 2 because the stakes are higher and the decisions less mechanical. Plan is to have Session 2 done over the next weekend, with the EC2 instance up but no software installed yet, before moving to Session 3 the weekend after.

**Open questions I'm carrying forward (things im thinking about in the middle of the night):**

- Whether the Metashape Python API on Linux behaves identically to the Windows version for the headless processing patterns I'll need in Session 5. The forum threads suggest yes, but I haven't tested it yet.
- Whether NICE DCV latency from East Coast US over residential broadband will be good enough for the precision UI work in Sessions 5 and 7. Worst case, I can do that work locally on the MacBook and sync results, but I'd prefer not to.
- **How structured to make the provenance layer, and whether to propose a portable
  format inspired by PFB.** This is the open question I'm most interested in,
  and it's worth saying more about because it's where my day-job background
  may actually contribute something the marine community doesn't have today.

  At my day job (NHLBI / BioData Catalyst) we use the Portable Format for
  Bioinformatics (PFB) — an Avro-based file format that bundles a schema,
  the metadata linking nodes and controlled-vocabulary references, and the
  data itself into a single transferable artifact. The point of PFB is that
  it makes biomedical data portable in a schema-aware way: you can hand
  someone a `.avro` file and they receive not just the data but the data
  dictionary that explains what the data means, the links between entities,
  and the controlled vocabularies (caDSR CDEs, ontology terms) that ground
  each field. The reference implementation is `pypfb`, the format is used
  to move datasets between Gen3 and analysis environments like
  Seven Bridges and Terra, and it works because the bioinformatics community
  has standardized data dictionaries and ontologies to plug into it.

  The Mote/USGS SfM pipeline — and as far as I can tell, reef SfM workflows
  more broadly — doesn't have anything like this. The Metashape processing
  report is an HTML file. Quality targets live in prose in methods papers.
  Derived metrics live in CSVs whose schemas are implicit. Benthic category
  vocabularies are project-specific. None of it is portable in a
  schema-aware way; if you want to integrate reconstructions from two
  programs, or compare a new run against published reference values, you're
  writing custom parsing every time.

  The provenance package in Session 6 is already structured around four
  concepts that map cleanly to a PFB-like schema:

  - **Intake records** — image-set inventories with hashes, EXIF, QC flags
  - **Processing manifests** — Metashape parameters, error metrics, software
    versions, input/output hashes, infrastructure identifiers
  - **QC validation records** — pass/fail evaluations against quality targets
    with criteria explicitly named (e.g. `reprojection_error_px`,
    `scale_bar_check_error_m`)
  - **Reconciled metric records** — derived topographic complexity values
    with explicit comparison to published reference values

  Each of these wants a schema, each wants to link to the others, and each
  would benefit from being serializable into a single portable artifact
  rather than living as a folder of loose JSON files. If I built this as a
  Portable Format for Reef SfM Provenance — name TBD, working title
  `PFRP` for now — using the same Avro + schema + metadata + data pattern
  PFB uses, I'd end up with:

  - A single `.avro` file per reconstruction containing the schema, the
    metadata about controlled vocabularies (where they exist), and the
    actual provenance data
  - Tools to `show` schema, metadata, and data the way `pfb show` works
  - A reference Python implementation (`pyreef-sfm-provenance` or similar)
  - A documented schema that another restoration program could adopt or
    extend

  The honest scoping question: building this for v1 is ambitious. The
  realistic v1 deliverable is a structured-JSON-with-explicit-schema
  implementation of the four record types, framed in the writeup as
  "PFB-inspired, designed to support a future portable Avro format." A
  v2 extension would actually build the Avro serialization and propose
  it as a community resource. Splitting the work this way protects the
  6–8 weekend MVP timeline while making the architectural intent visible.

  This is the question I want to keep revisiting through Sessions 4, 5,
  and 6, and that I expect will reshape Session 6 meaningfully. Worst
  case I produce a clean structured-JSON provenance layer and the PFB
  analogy is a framing device in the writeup. Best case it becomes the
  most reusable artifact of the project — a format that genuinely could
  be adopted by Mote, USGS, CRF, or any restoration program running
  similar SfM workflows, and that arrives in the field at a moment when
  no equivalent exists.

  Either way, the analogy itself is something I have specific
  professional grounding to draw. Most marine scientists don't work
  with PFB. Most data engineers don't work on reef restoration. Sitting
  at the intersection is, I think, exactly the place where the
  contribution gets made.

***

## Session Two — AWS Infrastructure Setup

**Date:** May 21, 2026
**Model used:** Claude Sonnet 4.6
**Status:** In progress — instance pending GPU quota approval
**Time:** ~3 hours active work, ~12 hours waiting on AWS

### What got built

The full AWS infrastructure layer — everything the EC2 workstation needs to exist, be reachable, and hold a Metashape license stably across stop/start cycles:

- **Key pair** (`reef-sfm-mote-keys-keypair`) — ED25519, saved to `~/.ssh/`, registered in AWS
- **Security group** (`reef-sfm-mote-keys-sg`) — SSH(22) and Amazon DCV(8443 TCP/UDP) inbound from my IP only, all other traffic blocked
- **Elastic IP** (`52.5.136.119`) — stable public address that survives instance stop/start
- **Secondary ENI** (`eni-0adf00e84bc402777`, MAC `02:3f:49:14:60:ab`) — dedicated network interface whose MAC will serve as Metashape's license fingerprint anchor
- **1 TB gp3 EBS data volume** (`vol-04b43c7a754f77852`) — standalone resource, not tied to the instance lifecycle, mounted at `/data`
- **Launch template** (`lt-0673a3230668b47f6`) — encodes the instance configuration with the DLAMI AMI ID pinned at build time (`ami-0f6205907e0adae99`, Deep Learning Base OSS Nvidia Driver GPU AMI Ubuntu 24.04, build 20260515)
- **IAM user** (`reef-sfm-cli`) — least-privilege user for CLI access, scoped to EC2 + SSM + Budgets, no admin credentials used for operational work
- **Monthly budget alert** — $100/month threshold, email notification

All infrastructure is version-controlled bash in `scripts/aws/`, not click-ops. Every script is idempotent.

The instance itself has not yet launched — blocked on an AWS vCPU quota increase request for `g6` instances in `us-east-1`. The request is open with AWS Support (case 1779299744500813). All other resources are sitting ready.

### What the AWS bureaucracy taught me

New AWS accounts start with a GPU vCPU quota of zero. This is not documented prominently anywhere. The process to fix it:

1. Submit a quota increase request via Service Quotas for "Running On-Demand G and VT instances" — the quota family that covers `g6` (NVIDIA L4), `g5` (A10G), and `g4dn` (T4) instance types
2. AWS opens a support case automatically and routes it to a team for human review
3. The initial "Your request has been validated" email refers to *account* validation, not *quota* approval — these are separate things and the naming is confusing
4. The quota increase itself requires a separate approval that can take hours to a day on a new account

The right number to request: 16 vCPUs (one g6.4xlarge). Don't ask for more than you need on a new account — larger requests get more scrutiny and the review takes longer.

For future projects: if you're planning GPU work on a new AWS account, submit the quota increase request a week before you need the instance, not the day of.

### The 8-step license stability pattern

The central operational constraint on this project is Metashape Professional's node-locked license. The license fingerprints against hardware identifiers — primarily the MAC address of a network interface. AWS's EC2 model creates a specific risk: if the underlying physical host changes (which can happen silently during stop/start cycles after a maintenance event), the primary ENI can get a new MAC and break the license fingerprint.

The mitigation is a secondary ENI attached at device-index 1. ENIs are independent AWS resources with stable MACs — the MAC lives with the ENI object, not with the instance or the underlying hardware. As long as the ENI itself is never deleted, the MAC is permanent. `start-instance.sh` verifies the MAC on every resume and bails loudly if it has changed, before Metashape ever runs.

The full 8-step pattern is documented in `docs/aws-setup.md`. In brief:
1. Never terminate and recreate the instance after Metashape activates
2. Keep the EBS root volume (stop, don't terminate)
3. Separate data volume on standalone EBS (survives termination)
4. Elastic IP (stable public address)
5. Fixed hostname (`reef-ec2`)
6. Secondary ENI for license MAC stability
7. No instance type changes after activation
8. Snapshots are reference points, not "same machine" equivalents

### Cost model

| State | Hourly rate | Monthly equivalent |
|---|---|---|
| Running | ~$1.32/hr compute + ~$0.13/hr EBS | ~$1,050 if never stopped |
| Stopped | $0 compute + ~$0.13/hr EBS | ~$96/month storage only |

The operational discipline is: stop the instance at the end of every session. Over a 6–8 weekend project (~168 active hours), expected total spend is $400–500. Forgetting to stop after one session costs ~$30–60; forgetting for a full week costs ~$220.

The data volume (1 TB gp3) started billing at creation, not at instance launch. Current ongoing cost: ~$80/month regardless of whether the instance is running.

### DLAMI choice and AMI pinning

The Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04) ships with:
- Ubuntu 24.04 LTS
- NVIDIA OSS driver (supports G6 / L4)  
- CUDA 12.8 toolkit
- No pre-installed ML frameworks (we don't need PyTorch or TensorFlow for Metashape)

The AMI ID is resolved at launch-template build time via an AWS SSM Parameter Store lookup and then pinned into the template. AWS publishes a new DLAMI every 1–2 weeks; using a live SSM lookup at instance launch would mean silently getting a different OS image. Pinning at template-build time gives reproducibility. The SSM parameter path that always resolves to the latest is:/aws/service/deeplearning/ami/x86_64/base-oss-nvidia-driver-gpu-ubuntu-24.04/latest/ami-id

Running `03-create-launch-template.sh` again adds a new template version with the current AMI, in case a security patch is needed later.

### Amazon DCV (formerly NICE DCV)

The product was rebranded from NICE DCV to Amazon DCV with the 2024.0 release. Current version is 2025.0-20103. The server installs on the EC2 instance in Session 3; the macOS client is already installed locally. Connection is on TCP/8443 (with optional UDP/8443 for QUIC acceleration). The security group opens both, restricted to my IP.

DCV will be used in two sessions:
- Session 5: Metashape GUI work (Gradual Selection error reduction, scale bar marker placement, manual quality review)
- Session 7: QGIS benthic annotation

Everything else runs over SSH.

### What's next

Waiting on AWS GPU quota approval. When it clears:

1. `./scripts/aws/04-launch-instance.sh` — launch, attach EIP/ENI/data volume, wait for status checks
2. Wire up `~/.ssh/config` from `config/ssh-config.snippet`
3. `./scripts/aws/05-first-boot-setup.sh` — format/mount data volume, set hostname, bring up secondary ENI
4. `./scripts/aws/06-create-baseline-snapshot.sh` — multi-volume snapshot before Session 3 touches anything
5. Stop the instance

Then Session 3: software bootstrap and Metashape trial activation. The trial clock does not start until the very last step of Session 3.

### Open questions carrying forward

- Whether the secondary ENI strategy actually matters in practice — Metashape will likely bind to the primary ENI's MAC (lowest device index), which is stable under the stop/start pattern as long as we never terminate. The secondary ENI is insurance against an AWS maintenance event rotating the primary MAC. I won't know whether I needed it until I don't need it.
- Amazon DCV latency over residential broadband for precision UI work. Will find out in Session 5.
- Whether the 30-day Metashape trial window is tight enough. Session 3 activates, Sessions 4–7 must produce real outputs before day 30. At one weekend per session that's 28 days of sessions — workable if nothing goes wrong.

In [1]:
# Session 2 resource inventory
# Auto-generated reference — do not edit manually
# Source: docs/aws-resources.md (gitignored, lives locally)

session2_resources = {
    "ami_id": "ami-0f6205907e0adae99",
    "ami_name": "Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04) 20260515",
    "instance_type": "g6.4xlarge",
    "region": "us-east-1",
    "availability_zone": "us-east-1a",
    "eip_public_ip": "52.5.136.119",
    "secondary_eni_id": "eni-0adf00e84bc402777",
    "secondary_eni_mac": "02:3f:49:14:60:ab",
    "data_volume_id": "vol-04b43c7a754f77852",
    "data_volume_size_gb": 1000,
    "launch_template_id": "lt-0673a3230668b47f6",
    "launch_template_version": 1,
    "security_group_id": "sg-0f252e1df4b0fd9af",
}

print("Session 2 infrastructure summary:")
for k, v in session2_resources.items():
    print(f"  {k}: {v}")

Session 2 infrastructure summary:
  ami_id: ami-0f6205907e0adae99
  ami_name: Deep Learning Base OSS Nvidia Driver GPU AMI (Ubuntu 24.04) 20260515
  instance_type: g6.4xlarge
  region: us-east-1
  availability_zone: us-east-1a
  eip_public_ip: 52.5.136.119
  secondary_eni_id: eni-0adf00e84bc402777
  secondary_eni_mac: 02:3f:49:14:60:ab
  data_volume_id: vol-04b43c7a754f77852
  data_volume_size_gb: 1000
  launch_template_id: lt-0673a3230668b47f6
  launch_template_version: 1
  security_group_id: sg-0f252e1df4b0fd9af
